# Cross-Partner Label Consistency Check

For each entity type (diseases, drugs, genes), load all three partners' map files,
find entities whose canonical URI appears in **two or more** partners, then verify
that every partner uses the **same label string** for the same URI.

A `✓` means all partners that share that entity agree on the label.  
A `✗` means at least two partners disagree — both labels are printed.

Run one section at a time. Diseases should only be run after the disease maps have
been refreshed with canonical MONDO labels (Radboud: after map_diseases-mondo.ipynb;
Demokritos: after the OLS4 patch cell).

In [13]:
require 'csv'
require 'set'

SAMPLE_SIZE = 30
srand(42)

# Build { canonical_id => label } from a CSV file.
# id_col:    column name to use as the join key
# label_col: column name whose value we're comparing
# normalize: optional lambda to clean up the id before use (e.g. extract CID number from URI)
def build_index(path, id_col:, label_col:, normalize: nil)
  index = {}
  CSV.foreach(path, headers: true) do |row|
    id    = row[id_col].to_s.strip
    id    = normalize.call(id) if normalize
    label = row[label_col].to_s.strip
    next if id.empty? || label.empty?
    index[id] ||= label   # first occurrence wins (handles multi-protein gene rows)
  end
  index
end

# Compare labels for a given entity type across three partner indexes.
# maps: { "Radboud" => index_hash, "Biovista" => index_hash, "Demokritos" => index_hash }
def cross_check(entity_type, maps)
  # Count how many partners have each ID
  id_partners = Hash.new { |h, k| h[k] = [] }
  maps.each { |partner, idx| idx.each_key { |id| id_partners[id] << partner } }

  overlapping = id_partners.select { |_, partners| partners.size >= 2 }.keys

  puts "#{entity_type}"
  puts "  Partners: #{maps.map { |p, idx| "#{p} (#{idx.size} entities)" }.join(', ')}"
  puts "  Overlapping IDs: #{overlapping.size}"

  if overlapping.empty?
    puts "  (no overlap found — nothing to compare)"
    puts
    return
  end

  sample = overlapping.sample([SAMPLE_SIZE, overlapping.size].min)
  puts "  Checking #{sample.size} randomly sampled overlapping entities:"
  puts

  matches = 0
  mismatches = 0

  sample.each do |id|
    present = maps.filter_map { |partner, idx| [partner, idx[id]] if idx.key?(id) }
    unique_labels = present.map { |_, l| l.downcase }.uniq

    if unique_labels.size == 1
      matches += 1
      puts "  ✓  [#{present.map { |p, _| p }.join(', ')}]  #{present.first[1]}"
    else
      mismatches += 1
      puts "  ✗  #{id}"
      present.each { |partner, label| puts "       #{partner.ljust(12)}: #{label}" }
    end
  end

  puts
  puts "  Result: #{matches}/#{sample.size} match  |  #{mismatches} mismatch(es)"
  puts
end

puts "Setup complete."


(irb):3: warning: already initialized constant Object::SAMPLE_SIZE
(irb):5: warning: previous definition of SAMPLE_SIZE was here


Setup complete.


---
## Diseases
Join key: MONDO URI — Label columns: `prefname` (Radboud, Demokritos), `name` (Biovista)

In [14]:
rad_dis = build_index('./radboud/maps/diseases.map',
                      id_col: 'mondo', label_col: 'prefname')

bv_dis  = build_index('./biovista/maps/2026-biovista-disease-mondo.map',
                      id_col: 'mondo', label_col: 'name')

dk_dis  = build_index('./demokritos/maps/2026-demokritos-disease-mondo.map',
                      id_col: 'mondo', label_col: 'prefname')

cross_check('Diseases', 'Radboud' => rad_dis, 'Biovista' => bv_dis, 'Demokritos' => dk_dis)


Diseases
  Partners: Radboud (5361 entities), Biovista (10 entities), Demokritos (3439 entities)
  Overlapping IDs: 868
  Checking 30 randomly sampled overlapping entities:

  ✓  [Radboud, Demokritos]  mitochondrial complex IV deficiency, nuclear type 11
  ✓  [Radboud, Demokritos]  Sandhoff disease, juvenile form
  ✓  [Radboud, Demokritos]  giardiasis
  ✓  [Radboud, Demokritos]  Holzgreve-Wagner-Rehder syndrome
  ✓  [Radboud, Demokritos]  mitochondrial complex IV deficiency, nuclear type 20
  ✓  [Radboud, Demokritos]  Kahrizi syndrome
  ✓  [Radboud, Demokritos]  ALG1-congenital disorder of glycosylation
  ✓  [Radboud, Demokritos]  perichondritis of auricle
  ✓  [Radboud, Demokritos]  obesity due to congenital leptin deficiency
  ✓  [Radboud, Demokritos]  purpura fulminans
  ✓  [Radboud, Demokritos]  Cenani-Lenz syndactyly syndrome
  ✓  [Radboud, Demokritos]  temtamy syndrome
  ✓  [Radboud, Demokritos]  sickle cell-hemoglobin c disease syndrome
  ✓  [Radboud, Demokritos]  Sturge-Weber s

---
## Drugs
Join key: PubChem CID number — Label column: `IUPACname` (all partners)

In [37]:
# Demokritos stores pubchem_cid as a full URI; extract the trailing number
cid_from_uri = ->(v) { v.match(/(\d+)\s*$/)&.[](1) || v }

rad_drug = build_index('./radboud/maps/drugs.map',
                       id_col: 'CID', label_col: 'IUPACname')

bv_drug  = build_index('./biovista/maps/2025-biovista-drugs.map',
                       id_col: 'CID', label_col: 'IUPACname')

dk_drug  = build_index('./demokritos/maps/2026-drug-mappings.map',
                       id_col: 'pubchem_cid', label_col: 'IUPACname',
                       normalize: cid_from_uri)

cross_check('Drugs', 'Radboud' => rad_drug, 'Biovista' => bv_drug, 'Demokritos' => dk_drug)


Drugs
  Partners: Radboud (1899 entities), Biovista (413 entities), Demokritos (668 entities)
  Overlapping IDs: 409
  Checking 30 randomly sampled overlapping entities:

  ✓  [Radboud, Biovista]  Dehydroepiandrosterone
  ✓  [Radboud, Biovista, Demokritos]  Carbidopa
  ✓  [Radboud, Demokritos]  Roflumilast
  ✓  [Radboud, Biovista, Demokritos]  Acetazolamide
  ✓  [Radboud, Demokritos]  Temozolomide
  ✓  [Radboud, Biovista, Demokritos]  Zonisamide
  ✓  [Radboud, Biovista]  Salicylic Acid
  ✓  [Radboud, Biovista, Demokritos]  Topiramate
  ✓  [Biovista, Demokritos]  Butyric Acid
  ✓  [Radboud, Biovista]  Prednisolone
  ✓  [Radboud, Biovista, Demokritos]  Metformin
  ✓  [Radboud, Demokritos]  Tyrosine
  ✓  [Radboud, Demokritos]  Pravastatin
  ✓  [Radboud, Demokritos]  Desmopressin
  ✗  3385
       Radboud     : 5-Fluorouracil
       Biovista    : Fluorouracil
       Demokritos  : Fluorouracil
  ✓  [Radboud, Demokritos]  Labetalol
  ✓  [Radboud, Biovista]  Pantothenic Acid
  ✗  5282411
     

---
## Genes
Join key: NCBI gene URI — Label column: `recommended_full` (all partners)

In [20]:
# Gene maps have multiple rows per geneid (one per UniProt accession).
# build_index takes the first occurrence, which is fine since recommended_full
# is the same for all rows sharing a geneid.

rad_gene = build_index('./radboud/maps/genes.map',
                       id_col: 'geneid', label_col: 'recommended_full')

bv_gene  = build_index('./biovista/maps/2025-biovista-genes.map',
                       id_col: 'geneid', label_col: 'recommended_full')

dk_gene  = build_index('./demokritos/maps/2026-gene-mappings.map',
                       id_col: 'geneid', label_col: 'recommended_full')

cross_check('Genes', 'Radboud' => rad_gene, 'Biovista' => bv_gene, 'Demokritos' => dk_gene)


Genes
  Partners: Radboud (728 entities), Biovista (575 entities), Demokritos (2377 entities)
  Overlapping IDs: 583
  Checking 30 randomly sampled overlapping entities:

  ✓  [Biovista, Demokritos]  Neuronal pentraxin-2
  ✓  [Biovista, Demokritos]  Asparaginyl-tRNA synthetase
  ✓  [Radboud, Demokritos]  Muscarinic acetylcholine receptor M2
  ✓  [Radboud, Demokritos]  Transient receptor potential cation channel subfamily V member 1
  ✓  [Biovista, Demokritos]  Peroxisomal membrane protein PEX13
  ✓  [Radboud, Demokritos]  Voltage-dependent T-type calcium channel subunit alpha-1G
  ✓  [Biovista, Demokritos]  Myc proto-oncogene protein
  ✓  [Radboud, Demokritos]  Coagulation factor V
  ✓  [Radboud, Demokritos]  Coagulation factor XII
  ✓  [Radboud, Demokritos]  Histone deacetylase 4
  ✓  [Biovista, Demokritos]  FAD-dependent oxidoreductase domain-containing protein 1
  ✓  [Radboud, Demokritos]  Sphingosine 1-phosphate receptor 3
  ✓  [Radboud, Biovista]  Tyrosine-protein kinase ITK/TSK
 